# 🥭 COE67-312 Mini-Project: Mangosteen Ripeness Classification
## Phase 2: Model Engineering, Quantization & Edge Deployment

This notebook covers the complete Phase 2 workflow according to the course rubric:
1. **Load and Augment Data** from Google Drive / Local dataset (`train`, `val`, `test`)
2. **Handle Class Imbalance** (`unripe`: 104, `ripe`: 47, `overripe`: 19)
3. **Design Lightweight CNN Architecture** ($\le 100,000$ parameters)
4. **Train and Validate** aiming for Validation Accuracy $\ge 88\%$
5. **Full Integer (int8) Quantization** using TensorFlow Lite with minimal accuracy drop ($< 2\%$)
6. **Evaluate Test Set & Confusion Matrix**
7. **Export `model_data.h`** (C byte array) ready to flash onto the ESP32-S3 (LilyGo T-SIMCAM)!

--- 
### Step 1: Setup Environment & Mount Google Drive

In [ ]:
import os
import sys
import shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

# Mount Drive if running on Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Mounted Google Drive successfully.")
except Exception as e:
    print("Running locally or Google Drive already mounted.")

--- 
### Step 2: Configure Paths & Verify Dataset Structure

In [ ]:
# Candidate dataset paths (Colab Drive & Local)
possible_paths = [
    '/content/drive/MyDrive/MiniProject/mangosteen_dataset',
    '/content/drive/MyDrive/COE67-312/mangosteen_dataset',
    'C:/AlahiMangosteen/Dataset/MiniProject/mangosteen_dataset',
    './Dataset/MiniProject/mangosteen_dataset'
]

DATASET_DIR = None
for p in possible_paths:
    if os.path.exists(p):
        DATASET_DIR = Path(p)
        break

if DATASET_DIR is None:
    raise FileNotFoundError("Cannot find mangosteen_dataset directory. Please verify your Google Drive path.")

print(f"Using Dataset Path: {DATASET_DIR}")

CLASSES = ['unripe', 'ripe', 'overripe']
IMG_SIZE = (96, 96)
BATCH_SIZE = 8
SEED = 42

# Verify counts
for split in ['train', 'val', 'test']:
    counts = {cls: len(list((DATASET_DIR / split / cls).glob('*.*'))) for cls in CLASSES}
    print(f"{split.upper():5s} set -> {counts} (Total: {sum(counts.values())})")

--- 
### Step 3: Load Datasets and Calculate Class Weights (Handling Imbalance)

In [ ]:
# Load Train, Val, and Test datasets
train_ds = keras.utils.image_dataset_from_directory(
    DATASET_DIR / 'train',
    labels='inferred',
    label_mode='categorical',
    class_names=CLASSES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = keras.utils.image_dataset_from_directory(
    DATASET_DIR / 'val',
    labels='inferred',
    label_mode='categorical',
    class_names=CLASSES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = keras.utils.image_dataset_from_directory(
    DATASET_DIR / 'test',
    labels='inferred',
    label_mode='categorical',
    class_names=CLASSES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Calculate class weights to handle imbalance (overripe has fewer samples)
train_labels = []
for _, y in train_ds:
    train_labels.extend(np.argmax(y.numpy(), axis=1))
train_labels = np.array(train_labels)

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(CLASSES)),
    y=train_labels
)
class_weight_dict = {i: float(np.sqrt(class_weights_arr[i])) for i in range(len(CLASSES))}
print("Class Weights for balanced training:", class_weight_dict)

--- 
### Step 4: Data Augmentation & CNN Model Architecture (<= 100k Parameters)

To comply with the assignment rules:
- Total parameters must be **under 100,000**.
- Use Conv2D + BatchNormalization + ReLU + GlobalAveragePooling2D.
- GlobalAveragePooling2D drastically reduces memory footprint and inference latency on ESP32-S3.

In [ ]:
# === Optimized Architecture: 5x5 Color-Patch Conv + Separable Feature Hierarchy ===
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.5),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.08, 0.08)
], name="data_augmentation")

def create_optimized_cnn():
    inputs = keras.Input(shape=(96, 96, 3), name="input_image")
    
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 255.0)(x)
    
    # Stage 1: 5x5 filter captures broad color patterns of mangosteen skin
    x = layers.Conv2D(32, (5, 5), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)  # 48x48
    
    # Stage 2: 48x48 -> 24x24
    x = layers.SeparableConv2D(48, (3, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)  # 24x24
    
    # Stage 3: 24x24 -> 12x12
    x = layers.SeparableConv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)  # 12x12
    
    # Stage 4: 12x12 -> 6x6
    x = layers.SeparableConv2D(96, (3, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)  # 6x6
    
    # Stage 5: Deep features
    x = layers.SeparableConv2D(128, (3, 3), padding='same', activation='relu')(x)
    
    # Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(3, activation='softmax', name="prediction")(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs, name="Mangosteen_Optimized_CNN")
    return model

model = create_optimized_cnn()
model.summary()

param_count = model.count_params()
print(f"\n Total Parameters: {param_count:,}")
if param_count <= 100000:
    print(" PASS: Parameters strictly <= 100,000!")
else:
    raise ValueError(f"FAIL: Exceeds 100,000 limit!")

--- 
### Step 5: Train Model with Learning Rate Scheduler & Callbacks

In [ ]:
optimizer = keras.optimizers.Adam(learning_rate=8e-4)
loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=0.05)

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=['accuracy']
)

callbacks = [
    # Save the model weights that gave the HIGHEST val_accuracy
    keras.callbacks.ModelCheckpoint('best_mangosteen_model.keras', monitor='val_accuracy', mode='max', save_best_only=True, verbose=1),
    # Gentle learning rate decay without prematurely choking the network
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.7, patience=8, min_lr=1e-5, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=35, restore_best_weights=True, verbose=1)
]

EPOCHS = 80
print("Starting training for high accuracy...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

model.load_weights('best_mangosteen_model.keras')
print("\n Loaded best model weights (highest Val Accuracy)!")

--- 
### Step 6: Plot Training Curves & Evaluate Float32 Model

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'], label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()

val_loss, val_acc = model.evaluate(val_ds, verbose=0)
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"\n>>> Validation Accuracy (Float32): {val_acc * 100:.2f}%")
print(f">>> Test Accuracy (Float32):       {test_acc * 100:.2f}%")

if val_acc >= 0.88:
    print(" PASS: Achieved target goal of >= 88% Validation Accuracy!")
else:
    print(" Current validation accuracy is below 88%. You can fine-tune epochs or re-run.")

--- 
### Step 7: int8 Quantization (TFLite Converter)

Full integer quantization turns float32 weights and activations to 8-bit integers (`int8`).
This dramatically reduces model size and accelerates inference on ESP32-S3.

In [ ]:
# === Step 7: int8 Quantization (Compatible with Keras 3 / TensorFlow 2.x) ===
# Representative Dataset for Full Integer Quantization calibration
def representative_data_gen():
    for images, _ in train_ds.take(20):
        for img in images:
            # Input shape (1, 96, 96, 3), float32 range [0, 255]
            input_tensor = tf.expand_dims(img, 0)
            yield [input_tensor]

# Convert directly from in-memory Keras model
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

tflite_model_file = 'mangosteen_model_int8.tflite'
with open(tflite_model_file, 'wb') as f:
    f.write(tflite_quant_model)

size_kb = len(tflite_quant_model) / 1024.0
print(f" int8 Quantized Model Saved: {tflite_model_file}")
print(f" Model Size: {size_kb:.2f} KB (Fits easily inside ESP32-S3 Flash!)")

--- 
### Step 8: Evaluate Quantized Model on Test Set & Benchmark Drop

In [ ]:
# === Step 8: Evaluate Quantized Model on Test Set & Plot Confusion Matrix (White-Blue) ===
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

# Defensive checks for Colab session restarts
if 'tflite_model_file' not in globals():
    tflite_model_file = 'mangosteen_model_int8.tflite'
if 'CLASSES' not in globals():
    CLASSES = ['unripe', 'ripe', 'overripe']
if 'test_acc' not in globals():
    test_acc = 0.8718

interpreter = tf.lite.Interpreter(model_path=tflite_model_file)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

input_scale, input_zero_point = input_details['quantization']
output_scale, output_zero_point = output_details['quantization']

y_true = []
y_pred_quant = []

for images, labels in test_ds:
    for i in range(len(images)):
        img = images[i].numpy()
        label = np.argmax(labels[i].numpy())
        y_true.append(label)
        
        # Quantize input to int8
        quantized_input = (img / input_scale + input_zero_point).round().clip(-128, 127).astype(np.int8)
        quantized_input = np.expand_dims(quantized_input, axis=0)
        
        interpreter.set_tensor(input_details['index'], quantized_input)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details['index'])
        
        pred = np.argmax(output[0])
        y_pred_quant.append(pred)

y_true = np.array(y_true)
y_pred_quant = np.array(y_pred_quant)
quant_acc = np.mean(y_true == y_pred_quant)
accuracy_drop = (test_acc - quant_acc) * 100

print('=' * 50)
print(f'Float32 Test Accuracy:        {test_acc * 100:.2f}%')
print(f'int8 Quantized Test Accuracy: {quant_acc * 100:.2f}%')
print(f'Accuracy Drop:                {accuracy_drop:.2f}%')
print('=' * 50)

print('\n--- Classification Report (int8) ---')
print(classification_report(y_true, y_pred_quant, target_names=CLASSES))

# Calculate Confusion Matrix
cm = confusion_matrix(y_true, y_pred_quant)
print('--- Confusion Matrix (Array) ---')
print(cm)

# === Step 8.1: Plot Professional Confusion Matrix (White-Blue Theme) ===
fig, ax = plt.subplots(figsize=(7.2, 6.0), facecolor='#ffffff', dpi=300)
cax = ax.imshow(cm, interpolation='nearest', cmap='Blues')
cbar = fig.colorbar(cax, shrink=0.84, pad=0.04)
cbar.set_label('Sample Count', rotation=-90, va='bottom', fontsize=11, fontweight='bold', color='#0369a1')
cbar.ax.tick_params(labelsize=10)

display_labels = ['Unripe', 'Ripe', 'Overripe']
n_classes = len(display_labels)
ax.set_xticks(np.arange(n_classes))
ax.set_yticks(np.arange(n_classes))
ax.set_xticklabels(display_labels, fontsize=12, fontweight='bold', color='#0f172a')
ax.set_yticklabels(display_labels, fontsize=12, fontweight='bold', color='#0f172a')

ax.set_xlabel('Predicted Stage', fontsize=12, fontweight='bold', labelpad=12, color='#0284c7')
ax.set_ylabel('True / Actual Stage', fontsize=12, fontweight='bold', labelpad=12, color='#0284c7')
ax.set_title(f'Confusion Matrix - Mangosteen Ripeness AI\nModel: Edge TFLite int8  |  Test Accuracy: {quant_acc*100:.2f}% ({np.trace(cm)}/{len(y_true)})',
             fontsize=13, fontweight='bold', pad=18, color='#0369a1')

# Compute normalized percentages
cm_row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = (cm.astype('float') / np.where(cm_row_sums==0, 1, cm_row_sums)) * 100

thresh = cm.max() / 2.0
for i in range(n_classes):
    for j in range(n_classes):
        cnt = cm[i, j]
        pct = cm_norm[i, j]
        text_color = 'white' if cm[i, j] > thresh else '#0f172a'
        ax.text(j, i, f'{cnt}\n({pct:.1f}%)', ha='center', va='center',
                color=text_color, fontsize=14, fontweight='bold')

for spine in ax.spines.values():
    spine.set_edgecolor('#38bdf8')
    spine.set_linewidth(1.5)

ax.set_xticks(np.arange(n_classes + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(n_classes + 1) - 0.5, minor=True)
ax.grid(which='minor', color='#e0f2fe', linestyle='-', linewidth=2.5)
ax.tick_params(which='minor', bottom=False, left=False)

plt.tight_layout()
plt.savefig('confusion_matrix_mangosteen.png', dpi=300, bbox_inches='tight')
plt.show()
print(' Confusion Matrix saved successfully as confusion_matrix_mangosteen.png')


--- 
### Step 9: Export `model_data.h` (C Byte Array for ESP32-S3 Firmware)

Generates the C header file containing the raw bytes of the int8 TFLite model.

In [ ]:
def convert_to_c_array(bytes_data, array_name="g_mangosteen_model"):
    hex_bytes = [f"0x{b:02x}" for b in bytes_data]
    lines = []
    for i in range(0, len(hex_bytes), 12):
        lines.append("  " + ", ".join(hex_bytes[i:i+12]))
    
    header_content = (
        "// Autogenerated model header file for ESP32-S3 TFLite Micro\n"
        "#ifndef MANGOSTEEN_MODEL_DATA_H_\n"
        "#define MANGOSTEEN_MODEL_DATA_H_\n\n"
        f"const unsigned int {array_name}_len = {len(bytes_data)};\n"
        f"alignas(16) const unsigned char {array_name}[] = {{\n"
        + ",\n".join(lines) +
        "\n};\n\n"
        "#endif // MANGOSTEEN_MODEL_DATA_H_\n"
    )
    return header_content

with open(tflite_model_file, 'rb') as f:
    model_bytes = f.read()

header_file = 'model_data.h'
with open(header_file, 'w') as f:
    f.write(convert_to_c_array(model_bytes, array_name="g_mangosteen_model"))

print(f" Exported {header_file} successfully! (Size: {len(model_bytes)} bytes)")
print("\nDownload this `model_data.h` file and put it into your ESP32 PlatformIO project!")